# Vail / Eagle River (VCRC2) Runoff Analysis

Two data sources:
- **Cell 1** — NOAA CBRFC Ensemble Streamflow Prediction (ESP) forecast for VCRC2
- **Cell 2** — SNODAS snowpack statistics for the VCRC2H basin

In [ ]:
# ── Cell 1: NOAA CBRFC ESP Forecast ──────────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import io

NOAA_URL = (
    "https://www.cbrfc.noaa.gov/wsup/graph/esptxt.py"
    "?id=VCRC2&year=2026&qpf=0&db=&csv=1"
)

resp = requests.get(NOAA_URL, timeout=30)
resp.raise_for_status()
raw = resp.text

# ── Parse ─────────────────────────────────────────────────────────────────────
# Format: a few metadata lines, then the header row starting with "Run Date"
# Columns: Run Date, max, p10, p30, p50, p70, p90, min, crx, c30, cmp, c70, crn, Obs
#   p10–p90  : ESP ensemble percentiles (remaining Apr–Jul volume, KAF)
#   max/min  : ensemble envelope
#   crx/c30/cmp/c70/crn : climatology reference (only on official monthly dates)
#   Obs      : cumulative observed volume (KAF) since season start
lines = raw.splitlines()
header_idx = next(i for i, ln in enumerate(lines) if ln.startswith("Run Date"))
df = pd.read_csv(io.StringIO("\n".join(lines[header_idx:])), na_values=["None", ""])
df["Run Date"] = pd.to_datetime(df["Run Date"])

df_esp   = df.dropna(subset=["p50"]).copy()   # rows with ESP forecast data
df_climo = df.dropna(subset=["cmp"]).copy()   # monthly official forecast dates
df_obs   = df.dropna(subset=["Obs"]).copy()   # rows with observed cumulative volume

print(f"ESP rows: {len(df_esp)}   Climatology dates: {len(df_climo)}   Obs rows: {len(df_obs)}")
print("Climatology dates:", df_climo["Run Date"].dt.strftime("%Y-%m-%d").tolist())
df_esp[["Run Date", "max", "p10", "p30", "p50", "p70", "p90", "min", "cmp", "Obs"]].head(10)


In [ ]:
# ── Plot ESP Forecast ─────────────────────────────────────────────────────────
has_obs  = len(df_obs) > 0
n_panels = 2 if has_obs else 1
fig, axes = plt.subplots(
    n_panels, 1, figsize=(13, 4.5 * n_panels),
    sharex=True, gridspec_kw={"hspace": 0.08}
)
if n_panels == 1:
    axes = [axes]
ax = axes[0]

x = df_esp["Run Date"]

# Outer band: p10–p90
ax.fill_between(x, df_esp["p10"], df_esp["p90"],
                color="steelblue", alpha=0.18, label="p10–p90 range")
# Inner band: p30–p70
ax.fill_between(x, df_esp["p30"], df_esp["p70"],
                color="steelblue", alpha=0.30, label="p30–p70 range")
# Ensemble envelope
ax.plot(x, df_esp["max"], color="gray", lw=0.8, ls="--", label="Max / Min")
ax.plot(x, df_esp["min"], color="gray", lw=0.8, ls="--")
# Median
ax.plot(x, df_esp["p50"], color="steelblue", lw=2.2, label="p50 (median)")

# Climatology at official monthly forecast dates (crx/c30/cmp/c70/crn)
if len(df_climo):
    ax.errorbar(
        df_climo["Run Date"], df_climo["cmp"],
        yerr=[df_climo["cmp"] - df_climo["crn"],
              df_climo["crx"] - df_climo["cmp"]],
        fmt="D", color="darkorange", ms=7, lw=1.5,
        capsize=5, zorder=5, label="Climo median ± full range"
    )

ax.set_title(
    "NOAA CBRFC ESP Runoff Forecast — VCRC2  (Apr–Jul 2026)\n"
    "Remaining seasonal volume (KAF)",
    fontsize=12, fontweight="bold"
)
ax.set_ylabel("Remaining Volume (KAF)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
ax.legend(loc="upper right", fontsize=8, ncol=2)
ax.grid(True, linestyle=":", alpha=0.5)

# Bottom panel: cumulative observed volume
if has_obs:
    ax2 = axes[1]
    ax2.plot(df_obs["Run Date"], df_obs["Obs"],
             color="black", lw=2, marker="o", ms=3, label="Cumulative observed")
    ax2.fill_between(df_obs["Run Date"], df_obs["Obs"], alpha=0.12, color="black")
    ax2.set_ylabel("Observed Vol. (KAF)")
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.1f}"))
    ax2.legend(loc="upper left", fontsize=8)
    ax2.grid(True, linestyle=":", alpha=0.5)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].set_xlabel("Run Date")
fig.autofmt_xdate(rotation=30, ha="right")
fig.tight_layout()
plt.show()


In [ ]:
# ── Cell 2: SNODAS Snowpack Statistics ───────────────────────────────────────
import requests
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import io

SNODAS_URL = (
    "https://snodas.cdss.state.co.us/data/SnowpackStatisticsByBasin/"
    "SnowpackStatisticsByBasin_VCRC2H_F.csv"
)

resp2 = requests.get(SNODAS_URL, timeout=30)
resp2.raise_for_status()

df_snow = pd.read_csv(io.StringIO(resp2.text))
df_snow.columns = df_snow.columns.str.strip()

print("Shape:", df_snow.shape)
print("Columns:", df_snow.columns.tolist())
df_snow.head(10)


In [ ]:
# ── Plot SNODAS Snowpack ──────────────────────────────────────────────────────
# Typical SNODAS columns (Colorado DWR format):
#   Date, LocalDate, SWE_Mean_in, SnowDepth_Mean_in, SWE_Volume_acft,
#   SnowCover_pct, SWE_PctOfMedian, SWE_PctOf30YrNormal, ...

ds = df_snow.copy()

# Identify date column
date_col = next(
    (c for c in ds.columns if "date" in c.lower() or "time" in c.lower()),
    ds.columns[0]
)
ds[date_col] = pd.to_datetime(ds[date_col], errors="coerce")
ds = ds.dropna(subset=[date_col]).sort_values(date_col)

# ── Helper: find column by keyword list ─────────────────────────────────────
def find_col(df, *keywords):
    for kw in keywords:
        match = next((c for c in df.columns if kw.lower() in c.lower()), None)
        if match:
            return match
    return None

swe_col    = find_col(ds, "swe", "snow_water", "snowwater")
depth_col  = find_col(ds, "depth", "snowdepth")
cover_col  = find_col(ds, "cover", "percent_covered", "pct")
pct_med    = find_col(ds, "pctofmedian", "pct_median", "median")
vol_col    = find_col(ds, "volume", "vol_acft", "acft")

# Build subplot grid based on what columns exist
panels = []
if swe_col:   panels.append((swe_col,   "SWE (in)",           "steelblue"))
if depth_col: panels.append((depth_col, "Snow Depth (in)",    "slateblue"))
if vol_col:   panels.append((vol_col,   "SWE Volume (ac-ft)", "teal"))
if pct_med:   panels.append((pct_med,   "% of Median SWE",    "darkorange"))
if cover_col: panels.append((cover_col, "Snow Cover (%)",     "mediumseagreen"))

# Fall back to all numeric columns if nothing matched
if not panels:
    num_cols = [c for c in ds.columns if c != date_col and pd.api.types.is_numeric_dtype(ds[c])]
    panels = [(c, c, "steelblue") for c in num_cols[:5]]

n_panels = len(panels)
fig, axes = plt.subplots(n_panels, 1, figsize=(12, 3.2 * n_panels), sharex=True)
if n_panels == 1:
    axes = [axes]

for ax, (col, ylabel, color) in zip(axes, panels):
    ax.plot(ds[date_col], ds[col], color=color, lw=1.5)
    ax.fill_between(ds[date_col], ds[col], alpha=0.15, color=color)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.1f}"))
    ax.grid(True, linestyle=":", alpha=0.5)

    # Draw 100% line for pct-of-median panel
    if "median" in ylabel.lower() or "normal" in ylabel.lower():
        ax.axhline(100, color="gray", lw=1, ls="--", label="100% of median")
        ax.legend(fontsize=8)

axes[0].set_title(
    "SNODAS Snowpack Statistics — VCRC2H Basin",
    fontsize=13, fontweight="bold"
)
axes[-1].set_xlabel("Date")
fig.tight_layout()
plt.show()
